# Split a combined suite2p recording

Split one suite2p run that contains **multiple recordings concatenated back-to-back** into a separate `suite2p/plane0` per recording, using explicit per-recording frame counts.

Wraps `utils.suite2p.split_suite2p_recordings`. It slices `F` / `Fneu` / `spks` (and `frames_time.npy` / `dFF.npy` if present) per recording and copies the shared cell metadata (`iscell` / `stat` / `ops` / `redcell`) into each output folder.

In [1]:
import sys, os
import numpy as np

# repo root must be on the path so `import utils` resolves
base_dir = r"C:\Users\acostamendoza\projects\distance-project"
if base_dir not in sys.path:
    sys.path.insert(0, base_dir)

from utils.suite2p import split_suite2p_recordings

## 1. Point at the combined run and check the total frame count

The split must add up to this number.

In [2]:
raw_base = r"D:\data-distance-coding-project\raw-data"
processed_base = r"D:\data-distance-coding-project\processed-data"

# the COMBINED suite2p run (recordings concatenated back-to-back)
joint_plane0_dir = os.path.join(raw_base, "260904_SAM_SAM045_goggles", "260904_SAM_SAM045_monocular-001", "suite2p", "plane0")

total = np.load(os.path.join(joint_plane0_dir, "F.npy")).shape[1]
print("total frames:", total)

total frames: 14320


## 2. Define the outputs and per-recording frame counts

`output_dirs` and `frame_counts` must be in **the same order the recordings were concatenated**, and `frame_counts` must sum to `total` (the function raises a `ValueError` otherwise).

In [ ]:
# one output plane0 per recording, IN CONCATENATION ORDER
output_dirs = [
    os.path.join(processed_base, "260904_SAM045_monocular", "suite2p", "plane0"),
    os.path.join(processed_base, "260904_SAM045_disparity", "suite2p", "plane0"),
    os.path.join(processed_base, "260904_SAM045_distance", "suite2p", "plane0")
]

# frames per recording (MUST sum to `total`)
frame_counts = [14432, 14432]   # <-- replace with your real per-recording counts
assert sum(frame_counts) == total, f"{sum(frame_counts)} != {total}"

### (Optional) read the frame counts from each recording's Bruker XML

Use this if you don't already know the per-recording counts.

In [ ]:
# import pandas as pd
# xmls = [r"D:\...\recording_1.xml", r"D:\...\recording_2.xml"]
# frame_counts = [len(pd.read_xml(x, xpath=".//Frame")) for x in xmls]
# print(frame_counts, "sum =", sum(frame_counts))
# assert sum(frame_counts) == total

## 3. Run the split

After this, each session folder loads directly with `DisparityDevice` / `ActivityLoader`.

In [15]:
split_suite2p_recordings(joint_plane0_dir, output_dirs, frame_counts)

Recording 1: 14432 frames → D:\data-distance-coding-project\processed-data\260903_SAM045_distance-temporal\suite2p\plane0
Recording 2: 14432 frames → D:\data-distance-coding-project\processed-data\260903_SAM045_distance-nasal\suite2p\plane0
Done. 1521 cells split into 2 recordings.
